# Decision Trees and Random Forest

Decision trees are intuitive, interpretable models that split data based on feature values. Random Forests combine many trees to create a more robust, accurate ensemble.

## Decision Tree Structure

A decision tree consists of:
- **Root node**: the topmost node representing the entire dataset
- **Internal nodes**: test a feature and branch based on the outcome
- **Leaf nodes**: terminal nodes that contain the prediction

### Splitting Criteria

**Gini Impurity** (for classification):
$$G = 1 - \sum_{i=1}^{c} p_i^2$$
where $p_i$ is the proportion of class $i$ in the node.

**Entropy**:
$$H = -\sum_{i=1}^{c} p_i \log_2(p_i)$$

**Information Gain** = Entropy(parent) - Weighted Avg(Entropy(children))

**Reduction in Variance** (for regression) — minimizes MSE within each leaf.

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris, load_wine, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_style('whitegrid')
print("Libraries imported.")

In [ ]:
# Load Iris dataset
iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
target_names = iris.target_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Iris dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Classes: {target_names}")

In [ ]:
# Decision Tree Classifier
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train, y_train)

y_pred = dt.predict(X_test)
train_acc = accuracy_score(y_train, dt.predict(X_train))
test_acc = accuracy_score(y_test, y_pred)

print(f"Decision Tree (max_depth=3):")
print(f"  Training accuracy: {train_acc:.4f}")
print(f"  Test accuracy:     {test_acc:.4f}")
print(f"  Number of nodes:   {dt.tree_.node_count}")

In [ ]:
# Visualize the Decision Tree
plt.figure(figsize=(16, 10))
plot_tree(dt, feature_names=feature_names, class_names=target_names,
          filled=True, rounded=True, fontsize=10)
plt.title('Decision Tree Visualization (Iris Dataset)', fontsize=14)
plt.show()

In [ ]:
# Feature Importance from Decision Tree
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': dt.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importances:")
print(importance_df.to_string(index=False))

# Plot
plt.figure(figsize=(8, 4))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='steelblue')
plt.xlabel('Importance')
plt.title('Decision Tree Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Effect of tree depth on overfitting
depths = range(1, 11)
train_scores = []
test_scores = []

for depth in depths:
    dt_temp = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt_temp.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, dt_temp.predict(X_train)))
    test_scores.append(accuracy_score(y_test, dt_temp.predict(X_test)))

plt.figure(figsize=(8, 4))
plt.plot(depths, train_scores, 'bo-', label='Training accuracy')
plt.plot(depths, test_scores, 'rs-', label='Test accuracy')
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.title('Overfitting: Deeper Trees Memorize Training Data')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print("Note: Test accuracy plateaus then may drop as tree overfits.")

## Pruning a Decision Tree

To prevent overfitting, we can:
1. **Pre-pruning**: limit `max_depth`, `min_samples_split`, `min_samples_leaf`
2. **Post-pruning** (cost-complexity pruning): use `ccp_alpha` parameter

Cost-complexity pruning finds the subtree that minimizes:
$$R_\alpha(T) = R(T) + \alpha|T|$$
where $R(T)$ is misclassification rate and $|T|$ is number of leaves.

In [ ]:
# Cost-complexity pruning
dt_unpruned = DecisionTreeClassifier(random_state=42)
path = dt_unpruned.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

dt_pruned = DecisionTreeClassifier(ccp_alpha=0.01, random_state=42)
dt_pruned.fit(X_train, y_train)

print(f"Unpruned tree nodes: {dt_unpruned.fit(X_train, y_train).tree_.node_count}")
print(f"Pruned tree nodes:   {dt_pruned.tree_.node_count}")
print(f"Pruned test accuracy: {accuracy_score(y_test, dt_pruned.predict(X_test)):.4f}")

In [ ]:
# Random Forest Classifier
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, oob_score=True)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
train_acc_rf = accuracy_score(y_train, rf.predict(X_train))
test_acc_rf = accuracy_score(y_test, y_pred_rf)

print("Random Forest Results:")
print(f"  Training accuracy: {train_acc_rf:.4f}")
print(f"  Test accuracy:     {test_acc_rf:.4f}")
print(f"  OOB score:         {rf.oob_score_:.4f}")
print(f"  Number of trees:   {rf.n_estimators}")
print(f"\n  vs Single Tree:")
print(f"  Single tree test:  {test_acc:.4f}")
print(f"  Random Forest:     {test_acc_rf:.4f}")

In [ ]:
# Compare Single Tree vs Forest — Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d',
            cmap='Blues', ax=axes[0], xticklabels=target_names, yticklabels=target_names)
axes[0].set_title('Single Decision Tree')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d',
            cmap='Greens', ax=axes[1], xticklabels=target_names, yticklabels=target_names)
axes[1].set_title('Random Forest (100 trees)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
# Random Forest Feature Importance (more reliable than single tree)
rf_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'DT_Importance': dt.feature_importances_,
    'RF_Importance': rf.feature_importances_
})

print("Feature Importance Comparison:")
print(rf_importance_df.to_string(index=False))

In [ ]:
# Wine dataset example
wine = load_wine()
X_w, y_w = wine.data, wine.target

X_w_train, X_w_test, y_w_train, y_w_test = train_test_split(
    X_w, y_w, test_size=0.2, random_state=42, stratify=y_w
)

# Decision Tree
dt_w = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_w_train, y_w_train)
dt_w_acc = accuracy_score(y_w_test, dt_w.predict(X_w_test))

# Random Forest
rf_w = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_w_train, y_w_train)
rf_w_acc = accuracy_score(y_w_test, rf_w.predict(X_w_test))

print("Wine Dataset (3 classes):")
print(f"  Decision Tree:  {dt_w_acc:.4f}")
print(f"  Random Forest:  {rf_w_acc:.4f}")
print(f"\n  Wine feature names: {wine.feature_names}")

In [ ]:
# Random Forest Regressor
from sklearn.datasets import load_diabetes
from sklearn.metrics import r2_score, mean_squared_error

diabetes = load_diabetes()
X_d, y_d = diabetes.data, diabetes.target

X_d_train, X_d_test, y_d_train, y_d_test = train_test_split(
    X_d, y_d, test_size=0.2, random_state=42
)

rf_reg = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
rf_reg.fit(X_d_train, y_d_train)

y_d_pred = rf_reg.predict(X_d_test)
print("Random Forest Regressor (Diabetes):")
print(f"  R² score:  {r2_score(y_d_test, y_d_pred):.4f}")
print(f"  RMSE:      {np.sqrt(mean_squared_error(y_d_test, y_d_pred)):.2f}")

In [ ]:
# Problem: Compare single tree vs forest on breast cancer
cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target
X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42, stratify=y_c
)

results = []
n_trees_options = [1, 5, 10, 50, 100, 200]

for n in n_trees_options:
    rf_temp = RandomForestClassifier(n_estimators=n, random_state=42)
    rf_temp.fit(X_c_train, y_c_train)
    results.append({
        'Trees': n,
        'Train Acc': accuracy_score(y_c_train, rf_temp.predict(X_c_train)),
        'Test Acc': accuracy_score(y_c_test, rf_temp.predict(X_c_test))
    })

results_df = pd.DataFrame(results)
print("Effect of Number of Trees:")
print(results_df.to_string(index=False))

plt.figure(figsize=(8, 4))
plt.plot(results_df['Trees'], results_df['Train Acc'], 'bo-', label='Training')
plt.plot(results_df['Trees'], results_df['Test Acc'], 'rs-', label='Test')
plt.xlabel('Number of Trees')
plt.ylabel('Accuracy')
plt.title('Random Forest: More Trees = More Stable')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Key Concepts: Bagging and Random Subspace

**Bagging (Bootstrap Aggregating)**:
- Each tree is trained on a bootstrap sample (random sample with replacement)
- Reduces variance without increasing bias

**Random Subspace**:
- At each split, only a random subset of features is considered
- Default: $\sqrt{n_{features}}$ for classification, $n_{features}/3$ for regression
- Increases diversity among trees

In [ ]:
# Interview Problem: Visualize tree structure and explain it
plt.figure(figsize=(20, 10))
plot_tree(dt, feature_names=feature_names, class_names=target_names,
          filled=True, rounded=True, fontsize=12, proportion=True)
plt.title('Trained Decision Tree — Iris Dataset')
plt.show()
print("Each node shows: split condition, samples, value (count per class),")

In [ ]:
# Interview Problem: When to use Decision Tree vs Random Forest?
print("""
Decision Trees:
- When you need INTERPRETABILITY (e.g., medical diagnosis explanation)
- When data is small and simple
- For quick baseline / sanity check
  
Random Forest:
- When you need HIGH ACCURACY
- When you have enough data
- When overfitting is a concern
- When you don't need deep interpretability

Tradeoffs:
- RF takes more memory and compute
- RF cannot be visualized easily
- Single trees are unstable (high variance) — RF solves this
""")

In [ ]:
# Summary
print("=" * 60)
print("DECISION TREES & RANDOM FOREST — KEY TAKEAWAYS")
print("=" * 60)
print("""
1. Trees split data based on feature values using Gini/Entropy
2. Pruning (max_depth, min_samples_leaf, ccp_alpha) prevents overfitting
3. Random Forest combines bagging + random feature subspace
4. RF reduces variance with minimal bias increase
5. feature_importances_ tells you which features matter most
6. Trees need NO feature scaling (threshold-based)
7. n_estimators: more trees = better (diminishing returns)
""")